# 01 — Data Quality Report

Runs the same checks as `tests/unit/test_dq_suite.py` against the full generated dataset and reports the numbers, rather than a pass/fail assertion. See `docs/data_dictionary.md` for table definitions and `docs/target_definition.md` for the point-in-time design.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from data import config

ROOT = '../data/raw'
customers = pd.read_parquet(f'{ROOT}/customers.parquet')
products = pd.read_parquet(f'{ROOT}/products.parquet')
orders = pd.read_parquet(f'{ROOT}/orders.parquet')
interactions = pd.read_parquet(f'{ROOT}/interactions.parquet')
support = pd.read_parquet(f'{ROOT}/support.parquet')
tables = {'customers': customers, 'products': products, 'orders': orders,
          'interactions': interactions, 'support': support}
{name: df.shape for name, df in tables.items()}

{'customers': (8000, 5),
 'products': (80, 3),
 'orders': (155010, 8),
 'interactions': (915894, 5),
 'support': (13117, 4)}

## Row counts, null rates, duplicate keys

In [2]:
pk = {'customers': 'customer_id', 'products': 'product_id', 'orders': 'order_id',
      'interactions': 'interaction_id', 'support': 'ticket_id'}
rows = []
for name, df in tables.items():
    key = pk[name]
    rows.append({
        'table': name,
        'rows': len(df),
        'duplicate_keys': int(df[key].duplicated().sum()),
        'any_fully_null_col': bool(df.isna().all().any()),
        'null_cells_pct': round(df.isna().mean().mean() * 100, 3),
    })
pd.DataFrame(rows)

,table,rows,duplicate_keys,any_fully_null_col,null_cells_pct
0,customers,8000,0,False,0.0
1,products,80,0,False,0.0
2,orders,155010,0,False,0.0
3,interactions,915894,0,False,0.0
4,support,13117,0,False,0.0


## Referential integrity

In [3]:
known_customers = set(customers['customer_id'])
known_products = set(products['product_id'])
print('orders.customer_id orphans:', (~orders['customer_id'].isin(known_customers)).sum())
print('orders.product_id orphans:', (~orders['product_id'].isin(known_products)).sum())
print('interactions.customer_id orphans:', (~interactions['customer_id'].isin(known_customers)).sum())
print('support.customer_id orphans:', (~support['customer_id'].isin(known_customers)).sum())

orders.customer_id orphans: 0
orders.product_id orphans: 0
interactions.customer_id orphans: 0
support.customer_id orphans: 0


## Value ranges and date integrity

In [4]:
print('order amount < 0:', (orders['amount'] < 0).sum())
print('order quantity <= 0:', (orders['quantity'] <= 0).sum())
print('discount outside [0,1]:', (~orders['discount'].between(0, 1)).sum())
print('unknown order status:', (~orders['status'].isin(['completed', 'refunded', 'cancelled'])).sum())
print('orders before DATA_START:', (orders['order_date'] < pd.Timestamp(config.DATA_START)).sum())
print('orders after DATA_END:', (orders['order_date'] > pd.Timestamp(config.DATA_END)).sum())
merged = orders.merge(customers[['customer_id', 'signup_date']], on='customer_id')
print('orders before customer signup (leakage):', (merged['order_date'] < merged['signup_date']).sum())

order amount < 0: 0
order quantity <= 0: 0
discount outside [0,1]: 0
unknown order status: 0
orders before DATA_START: 0
orders after DATA_END: 0
orders before customer signup (leakage): 0


## Order status breakdown

In [5]:
orders['status'].value_counts(normalize=True).round(4) * 100

status
completed    93.94
refunded      4.08
cancelled     1.98
Name: proportion, dtype: float64

## Verdict

All checks above should read zero orphans / zero out-of-range values / zero pre-signup orders — this is enforced by `tests/unit/test_dq_suite.py` in CI, this notebook is the human-readable version of the same checks for the data-quality report deliverable.